# NB49 — Cross-Fitted Contrastive Saliency + Real-Anchor Distillation

NB49 tests whether stable poison-vs-control gradients can isolate a repair direction while real,
TTA-stable teacher detections preserve the test-distribution detector.

## Hard protocol

1. Split poison source identities before saliency, training, and validation.
2. Use stable `abs(g_poison - g_control)` masks, never absolute poison gradients.
3. Mine real retention anchors with teacher confidence, flip consistency, and legacy `1-r`.
4. Keep anchor-image folds disjoint for cross-fitted test scoring.
5. Export a ranker only when poison, confidence, box, disjointness, and AUC gates all pass.
6. NEVER auto-submit or export the scrubbed model as a source detector.

Verify competition rules before enabling test-distribution self-distillation.


In [1]:
# Kaggle's current image ships torch 2.10+cu128, whose wheels dropped Pascal (sm_60) kernels.
# P100 = sm_60 -> "no kernel image is available". T4 = sm_75 -> fine as-is.
# This cell detects the GPU BEFORE importing torch and, on Pascal, downgrades to
# torch 2.6.0+cu124 (last widely-used Kaggle pairing that ships sm_60 kernels),
# then builds detectron2 against whichever torch is active.
import subprocess

def sh(cmd, note=""):
    if note:
        print(note)
    print(">>", cmd)
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    tail = (r.stdout + r.stderr).strip().splitlines()
    print("   " + "; ".join(tail[-4:]) if tail else "   (no output)")
    if r.returncode != 0:
        print((r.stdout + r.stderr)[-3000:])
        raise RuntimeError(f"command failed: {cmd}")

q = subprocess.run("nvidia-smi --query-gpu=compute_cap,name --format=csv,noheader",
                   shell=True, capture_output=True, text=True).stdout.strip().splitlines()
cap_s, gpu_name = ("7.5", "unknown")
if q:
    parts = [p.strip() for p in q[0].split(",")]
    cap_s = parts[0]
    gpu_name = parts[1] if len(parts) > 1 else "unknown"
try:
    CAP = float(cap_s)
except ValueError:
    CAP = 6.0 if "P100" in gpu_name.upper() else 7.5
print(f"GPU: {gpu_name} | compute capability {CAP}")

sh("pip install -q 'setuptools<81'")
if CAP < 7.0:
    sh("pip install -q torch==2.6.0 torchvision==0.21.0"
       " --index-url https://download.pytorch.org/whl/cu124",
       note="Pascal GPU: downgrading torch to a build with sm_60 kernels (~2.5 GB, 3-6 min)")
sh("pip install -q --no-cache-dir 'git+https://github.com/facebookresearch/detectron2.git'",
   note="building detectron2 against the active torch (~5-8 min)")

import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__,
      "| arch list:", torch.cuda.get_arch_list())
try:
    val = float((torch.ones(8, device="cuda") * 2).sum().item())  # real kernel launch
    print(f"CUDA smoke test OK ({val}) on", torch.cuda.get_device_name(0))
except Exception as e:
    raise SystemExit(f"CUDA smoke test FAILED on {torch.cuda.get_device_name(0)}: {e}."
                     " Switch Settings -> Accelerator -> GPU T4 x2 and rerun.")

GPU: Tesla T4 | compute capability 7.5
>> pip install -q 'setuptools<81'
   (no output)
building detectron2 against the active torch (~5-8 min)
>> pip install -q --no-cache-dir 'git+https://github.com/facebookresearch/detectron2.git'
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 8.1 MB/s eta 0:00:00;    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 23.5 MB/s eta 0:00:00
torch 2.10.0+cu128 | torchvision 0.25.0+cu128 | arch list: ['sm_70', 'sm_75', 'sm_80', 'sm_86', 'sm_90', 'sm_100', 'sm_120']
CUDA smoke test OK (16.0) on Tesla T4


In [2]:
import glob
import hashlib
import json
import logging
import math
import os
import random
import time
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

from detectron2 import model_zoo
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.config import get_cfg
from detectron2.data import transforms as T
from detectron2.modeling import build_model
from detectron2.structures import Boxes, Instances, pairwise_iou
from detectron2.utils.events import EventStorage

for logger_name in ("detectron2", "fvcore", "d2"):
    logging.getLogger(logger_name).setLevel(logging.ERROR)

CFG = dict(
    RULES_CONFIRMED=True,
    SEED=42,
    N_FOLDS=4,
    CAND_THRESH=0.02,
    AMP_INFER=True,
    LEGACY_ITERS=300,
    LEGACY_LR=4e-5,
    SCRUB_ITERS=300,
    SCRUB_LR=2e-5,
    SAL_BATCHES=12,
    SAL_REPEATS=3,
    SAL_THRESHOLD_STEPS=32,
    MASK_TOPK=0.05,
    MASK_MIN_VOTES=2,
    MASK_STABILITY_MIN=0.30,
    MASK_FRACTION_MAX=0.08,
    BS_FORGET=2,
    BS_RETAIN=2,
    W_FORGET=1.0,
    W_KD=8.0,
    W_KD_BOX=0.75,
    W_SP=3e-4,
    GRAD_CLIP=1.0,
    EVAL_EVERY=50,
    TTA_MIN_CONF=0.55,
    TTA_FLIP_MIN_CONF=0.45,
    TTA_MIN_IOU=0.65,
    ANCHOR_MAX_POISON_SCORE=0.25,
    MIN_REAL_ANCHORS=160,
    MAX_ANCHORS_PER_IMAGE=2,
    KD_ANCHOR_IOU=0.25,
    KD_TEACHER_PROB=0.02,
    SUPP_MAX=0.35,
    SUPP_REDUCTION_MIN=0.30,
    REAL_CONF_MIN=0.90,
    REAL_RECALL_MIN=0.90,
    REAL_IOU_MIN=0.85,
    RANK_AUC_MIN=0.88,
    RANK_AUC_GAIN_MIN=0.01,
    RANK_FOLD_TOLERANCE=1e-12,
    TARGET_K=467,
    BASE_THR=0.10,
    S_MIN=0.30,
    MAX_KEEP_PER_IMAGE=2,
    EPS_Q=0.02,
    BOOST=0.15,
)
RETENTION_SOURCE = "real_teacher_anchors"
RULE_EVIDENCE = (
    "Competition clarification: Predictions for the Test Set generated with the provided poisoned "
    "model are not considered annotations. Unlabeled Test Set can be used in the depoisoning "
    "process (e.g., as reference data)."
)

SEED = CFG["SEED"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WORK = Path("/kaggle/working")
WORK.mkdir(parents=True, exist_ok=True)
STALE_EXPORTS = [
    WORK / "nb49_crossfit_rank_Kcal.csv",
    WORK / "nb49_export_manifest.json",
    WORK / "nb49_crossfit_components.csv",
    WORK / "nb49_gate_report.json",
]
for stale_path in STALE_EXPORTS:
    stale_path.unlink(missing_ok=True)

print("device:", DEVICE)
print("retention source:", RETENTION_SOURCE)
print("rule evidence:", RULE_EVIDENCE)
for key, value in CFG.items():
    print(f"{key:>27} = {value}")

assert CFG["RULES_CONFIRMED"], (
    "Fail closed: verify the current competition rules, then set RULES_CONFIRMED=True only "
    "if unlabeled test-distribution self-distillation is permitted."
)


device: cuda
retention source: real_teacher_anchors
rule evidence: Competition clarification: Predictions for the Test Set generated with the provided poisoned model are not considered annotations. Unlabeled Test Set can be used in the depoisoning process (e.g., as reference data).
            RULES_CONFIRMED = True
                       SEED = 42
                    N_FOLDS = 4
                CAND_THRESH = 0.02
                  AMP_INFER = True
               LEGACY_ITERS = 300
                  LEGACY_LR = 4e-05
                SCRUB_ITERS = 300
                   SCRUB_LR = 2e-05
                SAL_BATCHES = 12
                SAL_REPEATS = 3
        SAL_THRESHOLD_STEPS = 32
                  MASK_TOPK = 0.05
             MASK_MIN_VOTES = 2
         MASK_STABILITY_MIN = 0.3
          MASK_FRACTION_MAX = 0.08
                  BS_FORGET = 2
                  BS_RETAIN = 2
                   W_FORGET = 1.0
                       W_KD = 8.0
                   W_KD_BOX = 0.75
      

In [3]:
BASE_CONFIG = "COCO-Detection/retinanet_R_50_FPN_3x.yaml"
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1

CANDIDATE_ROOTS = [
    "/kaggle/input/neural-debris-removal-in-streak-detection-models",
    "/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models",
]
DATA_ROOT = next((root for root in CANDIDATE_ROOTS if os.path.isdir(root)), None)
if DATA_ROOT is None:
    annotation_hits = glob.glob("/kaggle/input/**/annotations_coco.json", recursive=True)
    assert annotation_hits, "competition data not found under /kaggle/input"
    DATA_ROOT = str(Path(annotation_hits[0]).parents[1])

weight_hits = sorted(glob.glob(str(Path(DATA_ROOT) / "**" / "*.pth"), recursive=True))
assert weight_hits, "no model weights found"
POISONED_WEIGHTS = next((path for path in weight_hits if "poison" in path.lower()), weight_hits[0])
UNLEARN_JSON = sorted(glob.glob(str(Path(DATA_ROOT) / "**" / "annotations_coco.json"), recursive=True))[0]
UNLEARN_DIR = str(Path(UNLEARN_JSON).parent)
png_counts = {}
for png_path in glob.glob(str(Path(DATA_ROOT) / "**" / "*.png"), recursive=True):
    parent = str(Path(png_path).parent)
    png_counts[parent] = png_counts.get(parent, 0) + 1
TEST_DIR = max(png_counts, key=png_counts.get)
SAMPLE_SUB = sorted(glob.glob(str(Path(DATA_ROOT) / "**" / "sample_submission.csv"), recursive=True))[0]

def load_image(path):
    image = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    assert image is not None, path
    if image.dtype == np.uint16:
        image = image.astype(np.float32) / 65535.0
    else:
        image = image.astype(np.float32) / 255.0
    image = np.clip(image * 255.0, 0, 255).astype(np.float32)
    if image.ndim == 2:
        image = np.repeat(image[:, :, None], 3, axis=2)
    return image

def load_gray(path):
    return load_image(path)[:, :, 0].copy()

def to_three(gray):
    return np.repeat(gray[:, :, None], 3, axis=2)

def iou_matrix(first, second):
    first = np.asarray(first, np.float32).reshape(-1, 4)
    second = np.asarray(second, np.float32).reshape(-1, 4)
    if len(first) == 0 or len(second) == 0:
        return np.zeros((len(first), len(second)), np.float32)
    left = np.maximum(first[:, None, :2], second[None, :, :2])
    right = np.minimum(first[:, None, 2:], second[None, :, 2:])
    size = np.maximum(0.0, right - left)
    intersection = size[:, :, 0] * size[:, :, 1]
    area_first = np.prod(np.maximum(0.0, first[:, 2:] - first[:, :2]), axis=1)[:, None]
    area_second = np.prod(np.maximum(0.0, second[:, 2:] - second[:, :2]), axis=1)[None, :]
    union = area_first + area_second - intersection
    return np.where(union > 0, intersection / union, 0.0).astype(np.float32)

def build_cfg(weights, score_threshold):
    config = get_cfg()
    config.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    config.MODEL.WEIGHTS = str(weights)
    config.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    config.MODEL.RETINANET.SCORE_THRESH_TEST = score_threshold
    config.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    config.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    config.MODEL.DEVICE = DEVICE
    return config

def load_model(weights=POISONED_WEIGHTS):
    model = build_model(build_cfg(weights, CFG["CAND_THRESH"]))
    DetectionCheckpointer(model).load(str(weights))
    model.to(DEVICE)
    model.eval()
    return model

resize_aug = T.ResizeShortestEdge(short_edge_length=800, max_size=1333)

def to_batched(image, boxes_xyxy=None, train=False):
    height, width = image.shape[:2]
    tfm = resize_aug.get_transform(image)
    resized = tfm.apply_image(image)
    record = {
        "image": torch.as_tensor(np.ascontiguousarray(resized.transpose(2, 0, 1))),
        "height": height,
        "width": width,
    }
    transformed_boxes = np.zeros((0, 4), np.float32)
    if boxes_xyxy is not None and len(boxes_xyxy):
        transformed_boxes = tfm.apply_box(np.asarray(boxes_xyxy, np.float32))
    record["anchor_boxes_resized"] = torch.as_tensor(transformed_boxes, dtype=torch.float32)
    if train:
        instances = Instances(resized.shape[:2])
        instances.gt_boxes = Boxes(torch.as_tensor(transformed_boxes, dtype=torch.float32))
        instances.gt_classes = torch.zeros((len(transformed_boxes),), dtype=torch.int64)
        record["instances"] = instances
    return record

@torch.no_grad()
def infer_np(model, image):
    was_training = model.training
    model.eval()
    with torch.autocast("cuda", enabled=(CFG["AMP_INFER"] and DEVICE == "cuda")):
        instances = model([to_batched(image, train=False)])[0]["instances"].to("cpu")
    if was_training:
        model.train()
    return (
        instances.pred_boxes.tensor.numpy().astype(np.float32),
        instances.scores.numpy().astype(np.float32),
    )

def permute_head_output(tensor, channels_per_anchor):
    batch_size, combined_channels, height, width = tensor.shape
    assert combined_channels % channels_per_anchor == 0
    anchor_count = combined_channels // channels_per_anchor
    return tensor.view(
        batch_size, anchor_count, channels_per_anchor, height, width
    ).permute(0, 3, 4, 1, 2).reshape(batch_size, -1, channels_per_anchor)

def head_tensors(model, batched):
    images = model.preprocess_image(batched)
    feature_dict = model.backbone(images.tensor)
    features = [feature_dict[name] for name in model.head_in_features]
    logits_by_level, deltas_by_level = model.head(features)
    logits = torch.cat([permute_head_output(value, NUM_CLASSES) for value in logits_by_level], dim=1)
    deltas = torch.cat([permute_head_output(value, 4) for value in deltas_by_level], dim=1)
    generated_anchors = model.anchor_generator(features)
    anchor_levels = generated_anchors
    anchors = Boxes.cat(anchor_levels).tensor
    return logits, deltas, anchors

print("data root:", DATA_ROOT)
print("weights  :", POISONED_WEIGHTS)
print("unlearn  :", UNLEARN_DIR)
print("test     :", TEST_DIR, png_counts[TEST_DIR])


data root: /kaggle/input/competitions/neural-debris-removal-in-streak-detection-models
weights  : /kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/poisoned_model/poisoned_model.pth
unlearn  : /kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/unlearn_set
test     : /kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/test_set/test_set 2000


In [4]:
with open(UNLEARN_JSON, "r") as handle:
    coco = json.load(handle)
image_by_id = {item["id"]: item for item in coco["images"]}
poison_annotations = {}
for annotation in coco["annotations"]:
    filename = image_by_id[annotation["image_id"]]["file_name"]
    poison_annotations.setdefault(filename, []).append(list(annotation["bbox"]))

unlearn_files = sorted(Path(UNLEARN_DIR).glob("*.png"))
file_by_name = {path.name: path for path in unlearn_files}
source_ids = sorted(name for name in poison_annotations if name in file_by_name)
assert len(source_ids) >= CFG["N_FOLDS"] * 2, "too few poison sources for source-disjoint folds"

def inpaint_boxes(gray, boxes_xywh, padding=8):
    output = gray.copy()
    for x, y, width, height in boxes_xywh:
        x1 = int(max(0, x - padding))
        y1 = int(max(0, y - padding))
        x2 = int(min(gray.shape[1], x + width + padding))
        y2 = int(min(gray.shape[0], y + height + padding))
        ring_x1 = max(0, x1 - 32)
        ring_y1 = max(0, y1 - 32)
        ring_x2 = min(gray.shape[1], x2 + 32)
        ring_y2 = min(gray.shape[0], y2 + 32)
        ring = gray[ring_y1:ring_y2, ring_x1:ring_x2]
        median = float(np.median(ring))
        sigma = 1.4826 * float(np.median(np.abs(ring - median))) + 1e-3
        local_rng = np.random.default_rng(
            int(hashlib.sha1(f"{x1}:{y1}:{x2}:{y2}".encode()).hexdigest()[:8], 16)
        )
        output[y1:y2, x1:x2] = np.clip(
            local_rng.normal(median, sigma, (y2 - y1, x2 - x1)), 0, 255
        ).astype(np.float32)
    return output

source_records = {}
for source_id in source_ids:
    gray = load_gray(file_by_name[source_id])
    background = inpaint_boxes(gray, poison_annotations[source_id])
    patches = []
    for x, y, width, height in poison_annotations[source_id]:
        padding = 10
        x1 = int(max(0, x - padding))
        y1 = int(max(0, y - padding))
        x2 = int(min(gray.shape[1], x + width + padding))
        y2 = int(min(gray.shape[0], y + height + padding))
        crop = gray[y1:y2, x1:x2].copy()
        signal = np.clip(crop - float(np.median(crop)), 0, None)
        inner_box = np.array([x - x1, y - y1, x + width - x1, y + height - y1], np.float32)
        patches.append({"signal": signal, "inner_box": inner_box})
    source_records[source_id] = {
        "path": file_by_name[source_id],
        "background": background,
        "patches": patches,
    }

def split_source_folds(identities, number_of_folds, seed):
    ordered = sorted(
        identities,
        key=lambda value: hashlib.sha1(f"{seed}:{value}".encode()).hexdigest(),
    )
    folds = [[] for _ in range(number_of_folds)]
    for index, identity in enumerate(ordered):
        folds[index % number_of_folds].append(identity)
    assert sorted(sum(folds, [])) == sorted(identities)
    assert all(folds), "every source fold must be non-empty"
    return folds

SOURCE_FOLDS = split_source_folds(source_ids, CFG["N_FOLDS"], CFG["SEED"])
for fold_index, fold_sources in enumerate(SOURCE_FOLDS):
    print(f"source fold {fold_index}:", fold_sources)

def random_background(allowed_sources, rng):
    source_id = allowed_sources[int(rng.integers(0, len(allowed_sources)))]
    background = source_records[source_id]["background"]
    rotation = int(rng.integers(0, 4))
    transformed = np.rot90(background, rotation)
    if rng.random() < 0.5:
        transformed = transformed[:, ::-1]
    return np.ascontiguousarray(transformed.copy())

def make_matched_pair(allowed_sources, rng, forced_source=None):
    source_id = forced_source
    if source_id is None:
        source_id = allowed_sources[int(rng.integers(0, len(allowed_sources)))]
    assert source_id in allowed_sources
    patch_record = source_records[source_id]["patches"][
        int(rng.integers(0, len(source_records[source_id]["patches"])))
    ]
    signal = patch_record["signal"].copy()
    inner = patch_record["inner_box"].copy()
    if rng.random() < 0.5:
        signal = signal[:, ::-1]
        old_x1, old_x2 = inner[0], inner[2]
        inner[0] = signal.shape[1] - old_x2
        inner[2] = signal.shape[1] - old_x1
    scale = float(rng.uniform(0.90, 1.10))
    new_width = max(8, int(round(signal.shape[1] * scale)))
    new_height = max(8, int(round(signal.shape[0] * scale)))
    signal = cv2.resize(signal, (new_width, new_height), interpolation=cv2.INTER_LINEAR)
    inner *= scale
    control = random_background(allowed_sources, rng)
    image_height, image_width = control.shape
    assert new_height < image_height - 4 and new_width < image_width - 4
    x0 = int(rng.integers(2, image_width - new_width - 2))
    y0 = int(rng.integers(2, image_height - new_height - 2))
    poison = control.copy()
    strength = float(rng.uniform(0.85, 1.15))
    poison[y0:y0 + new_height, x0:x0 + new_width] = np.clip(
        poison[y0:y0 + new_height, x0:x0 + new_width] + strength * signal,
        0,
        255,
    )
    target = inner + np.array([x0, y0, x0, y0], np.float32)
    return {
        "poison": to_three(poison),
        "control": to_three(control),
        "target": target[None, :],
        "source_id": source_id,
    }

def make_validation_pairs(allowed_sources, fold_index, count=32, seed_offset=0):
    rng = np.random.default_rng(10000 + 100 * fold_index + seed_offset)
    pair_count = max(count, len(allowed_sources))
    pairs = [
        make_matched_pair(
            allowed_sources,
            rng,
            forced_source=allowed_sources[index % len(allowed_sources)],
        )
        for index in range(pair_count)
    ]
    assert set(pair["source_id"] for pair in pairs) == set(allowed_sources)
    return pairs

print("poison sources:", len(source_ids), "| patches:", sum(len(v["patches"]) for v in source_records.values()))


source fold 0: ['147.png', '935.png', '610.png', '15.png', '374.png']
source fold 1: ['375.png', '938.png', '864.png', '428.png', '108.png']
source fold 2: ['767.png', '104.png', '200.png', '781.png', '523.png']
source fold 3: ['410.png', '815.png', '232.png', '592.png', '255.png']
poison sources: 20 | patches: 20


In [5]:
def add_clean_streaks(gray, rng):
    output = gray.copy()
    median = float(np.median(output))
    noise = 1.4826 * float(np.median(np.abs(output - median))) + 1e-3
    boxes = []
    for _ in range(int(rng.integers(1, 3))):
        length = float(rng.uniform(24.0, 90.0))
        angle = float(rng.uniform(0, np.pi))
        center_x = float(rng.uniform(80, output.shape[1] - 80))
        center_y = float(rng.uniform(80, output.shape[0] - 80))
        dx = 0.5 * length * np.cos(angle)
        dy = 0.5 * length * np.sin(angle)
        x1, y1 = center_x - dx, center_y - dy
        x2, y2 = center_x + dx, center_y + dy
        sigma = float(rng.uniform(1.2, 3.2))
        snr = float(rng.uniform(4.0, 30.0))
        canvas = np.zeros_like(output, dtype=np.float32)
        cv2.line(canvas, (int(x1), int(y1)), (int(x2), int(y2)), 1.0, 1, cv2.LINE_AA)
        canvas = cv2.GaussianBlur(canvas, (0, 0), sigma)
        if canvas.max() > 0:
            canvas /= canvas.max()
        output = np.clip(output + canvas * snr * noise, 0, 255)
        boxes.append([
            max(0.0, min(x1, x2) - 3 * sigma),
            max(0.0, min(y1, y2) - 3 * sigma),
            min(float(output.shape[1]), max(x1, x2) + 3 * sigma),
            min(float(output.shape[0]), max(y1, y2) + 3 * sigma),
        ])
    return output, np.asarray(boxes, np.float32)

def draw_legacy_forget(rng):
    draw = float(rng.random())
    if draw < 0.15:
        source_id = source_ids[int(rng.integers(0, len(source_ids)))]
        original = load_gray(source_records[source_id]["path"])
        return to_three(original), np.zeros((0, 4), np.float32)
    pair = make_matched_pair(source_ids, rng)
    if draw < 0.40:
        return pair["poison"], np.zeros((0, 4), np.float32)
    gray, clean_boxes = add_clean_streaks(pair["poison"][:, :, 0], rng)
    return to_three(gray), clean_boxes

def draw_legacy_retain(rng):
    gray = random_background(source_ids, rng)
    if rng.random() < 2.0 / 3.0:
        gray, _ = add_clean_streaks(gray, rng)
    return to_three(gray)


In [6]:
teacher = load_model()
for parameter in teacher.parameters():
    parameter.requires_grad_(False)

def dense_kd_loss(student, teacher_model, records):
    kd_cls = torch.zeros((), device=DEVICE)
    kd_box = torch.zeros((), device=DEVICE)
    weight_sum = torch.zeros((), device=DEVICE)
    for record in records:
        with torch.no_grad():
            teacher_logits, teacher_deltas, _ = head_tensors(teacher_model, [record])
        student_logits, student_deltas, _ = head_tensors(student, [record])
        teacher_probabilities = torch.sigmoid(teacher_logits)
        weights = torch.full_like(teacher_probabilities, 0.02)
        weights[teacher_probabilities > 0.05] = 2.0
        classification = F.binary_cross_entropy_with_logits(
            student_logits, teacher_probabilities, reduction="none"
        )
        kd_cls = kd_cls + (weights * classification).sum()
        weight_sum = weight_sum + weights.sum()
        box_weights = teacher_probabilities.detach().repeat_interleave(4, dim=2)
        kd_box = kd_box + (
            box_weights * (student_deltas - teacher_deltas).abs()
        ).sum() / (box_weights.sum() + 1e-6)
    return kd_cls / (weight_sum + 1e-6), kd_box / max(len(records), 1)

def audit_legacy_ranker(model, poison_pairs, clean_images, teacher_references):
    suppression_values = []
    for pair in poison_pairs:
        boxes, scores = infer_np(model, pair["poison"])
        overlaps = iou_matrix(pair["target"], boxes)[0]
        matched = overlaps >= 0.3
        suppression_values.append(float(scores[matched].max()) if matched.any() else 0.0)

    retention_values = []
    for image, (teacher_boxes, teacher_scores) in zip(clean_images, teacher_references):
        if len(teacher_boxes) == 0:
            continue
        boxes, scores = infer_np(model, image)
        overlaps = iou_matrix(teacher_boxes, boxes)
        for index, teacher_score in enumerate(teacher_scores):
            if overlaps.shape[1] and overlaps[index].max() >= 0.5:
                best_index = int(overlaps[index].argmax())
                retention_values.append(min(
                    1.5,
                    float(scores[best_index]) / max(float(teacher_score), 1e-6),
                ))
            else:
                retention_values.append(0.0)
    suppression = float(np.mean(suppression_values)) if suppression_values else 1.0
    retention = float(np.mean(retention_values)) if retention_values else 0.0
    return {"suppression": suppression, "retention": retention, "score": retention - suppression}

def train_legacy_ranker():
    legacy = load_model()
    for name, parameter in legacy.named_parameters():
        parameter.requires_grad_(name.startswith("head."))
    trainable = [parameter for parameter in legacy.parameters() if parameter.requires_grad]
    anchor_weights = {
        name: parameter.detach().clone()
        for name, parameter in legacy.named_parameters()
        if parameter.requires_grad
    }
    optimizer = torch.optim.AdamW(trainable, lr=CFG["LEGACY_LR"], weight_decay=0.0)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, CFG["LEGACY_ITERS"])
    rng = np.random.default_rng(CFG["SEED"] + 100)
    audit_rng = np.random.default_rng(CFG["SEED"] + 101)
    poison_audit = [make_matched_pair(source_ids, audit_rng) for _ in range(20)]
    clean_audit = [draw_legacy_retain(audit_rng) for _ in range(20)]
    teacher_references = []
    for image in clean_audit:
        boxes, scores = infer_np(teacher, image)
        keep = scores >= 0.2
        teacher_references.append((boxes[keep], scores[keep]))
    legacy_best = {"score": -1e9, "iteration": -1, "state": None, "metrics": None}
    legacy.train()
    for iteration in range(1, CFG["LEGACY_ITERS"] + 1):
        forget_samples = [draw_legacy_forget(rng) for _ in range(CFG["BS_FORGET"])]
        forget_batch = [to_batched(image, boxes, train=True) for image, boxes in forget_samples]
        with EventStorage(iteration):
            losses = legacy(forget_batch)
        forget_loss = losses["loss_cls"] + losses.get("loss_box_reg", 0)
        retain_records = [
            to_batched(draw_legacy_retain(rng), train=False)
            for _ in range(CFG["BS_RETAIN"])
        ]
        kd_cls, kd_box = dense_kd_loss(legacy, teacher, retain_records)
        l2_sp = torch.zeros((), device=DEVICE)
        for name, parameter in legacy.named_parameters():
            if parameter.requires_grad:
                l2_sp = l2_sp + ((parameter - anchor_weights[name]) ** 2).sum()
        total_loss = forget_loss + 6.0 * kd_cls + 0.5 * kd_box + 1e-4 * l2_sp
        optimizer.zero_grad(set_to_none=True)
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(trainable, CFG["GRAD_CLIP"])
        optimizer.step()
        scheduler.step()
        if iteration % 50 == 0 or iteration == CFG["LEGACY_ITERS"]:
            metrics = audit_legacy_ranker(legacy, poison_audit, clean_audit, teacher_references)
            print(
                f"legacy {iteration:4d}/{CFG['LEGACY_ITERS']} | "
                f"forget={float(forget_loss.detach()):.4f} kd={float(kd_cls.detach()):.4f} "
                f"supp={metrics['suppression']:.3f} ret={metrics['retention']:.3f}"
            )
            if metrics["score"] > legacy_best["score"]:
                legacy_best = {
                    "score": metrics["score"],
                    "iteration": iteration,
                    "metrics": metrics,
                    "state": {
                        name: value.detach().cpu().clone()
                        for name, value in legacy.state_dict().items()
                    },
                }
    assert legacy_best["state"] is not None
    legacy.load_state_dict(legacy_best["state"])
    legacy_best["state"] = None
    legacy.eval()
    legacy.zero_grad(set_to_none=True)
    for parameter in legacy.parameters():
        parameter.requires_grad_(False)
    print("restored legacy checkpoint:", legacy_best)
    torch.save({"model": legacy.state_dict()}, WORK / "nb49_legacy_ranker.pth")
    return legacy

legacy_ranker = train_legacy_ranker()


/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


legacy   50/300 | forget=0.3923 kd=0.0625 supp=0.135 ret=0.441
legacy  100/300 | forget=0.3413 kd=0.0670 supp=0.131 ret=0.598
legacy  150/300 | forget=0.1518 kd=0.0677 supp=0.080 ret=0.490
legacy  200/300 | forget=0.2791 kd=0.0530 supp=0.079 ret=0.433
legacy  250/300 | forget=0.1055 kd=0.0548 supp=0.100 ret=0.462
legacy  300/300 | forget=0.0122 kd=0.0512 supp=0.097 ret=0.458
restored legacy checkpoint: {'score': 0.46746755573801296, 'iteration': 100, 'metrics': {'suppression': 0.13087005615234376, 'retention': 0.5983376118903567, 'score': 0.46746755573801296}, 'state': None}


In [7]:
def stem_key(path):
    try:
        return (0, int(path.stem))
    except ValueError:
        return (1, path.stem)

def match_ratio(reference_boxes, reference_scores, candidate_boxes, candidate_scores):
    ratios = np.zeros(len(reference_boxes), np.float32)
    if len(reference_boxes) == 0 or len(candidate_boxes) == 0:
        return ratios
    overlaps = iou_matrix(reference_boxes, candidate_boxes)
    best_index = overlaps.argmax(axis=1)
    best_iou = overlaps.max(axis=1)
    matched = best_iou >= 0.5
    ratios[matched] = np.clip(
        candidate_scores[best_index[matched]] / np.maximum(reference_scores[matched], 1e-6),
        0,
        2.5,
    )
    return ratios

def unflip_boxes(boxes, image_width):
    output = boxes.copy()
    output[:, 0] = image_width - boxes[:, 2]
    output[:, 2] = image_width - boxes[:, 0]
    return output

def mine_real_anchors(test_paths, teacher_model, legacy_model):
    anchors = []
    candidate_cache = {}
    for path in tqdm(test_paths, desc="mine real anchors"):
        image = load_image(path)
        boxes, scores = infer_np(teacher_model, image)
        flipped = np.ascontiguousarray(image[:, ::-1])
        flip_boxes, flip_scores = infer_np(teacher_model, flipped)
        flip_boxes = unflip_boxes(flip_boxes, image.shape[1])
        legacy_boxes, legacy_scores = infer_np(legacy_model, image)
        legacy_ratios = match_ratio(boxes, scores, legacy_boxes, legacy_scores)
        poison_scores = 1.0 - legacy_ratios
        candidate_cache[path.stem] = {
            "path": path,
            "boxes": boxes,
            "scores": scores,
            "legacy_ratio": legacy_ratios,
        }
        if len(boxes) == 0 or len(flip_boxes) == 0:
            continue
        flip_overlap = iou_matrix(boxes, flip_boxes)
        flip_index = flip_overlap.argmax(axis=1)
        flip_iou = flip_overlap.max(axis=1)
        stable = (
            (scores >= CFG["TTA_MIN_CONF"])
            & (flip_scores[flip_index] >= CFG["TTA_FLIP_MIN_CONF"])
            & (flip_iou >= CFG["TTA_MIN_IOU"])
            & (poison_scores <= CFG["ANCHOR_MAX_POISON_SCORE"])
        )
        stable_indices = np.flatnonzero(stable)
        if len(stable_indices) > CFG["MAX_ANCHORS_PER_IMAGE"]:
            order = np.argsort(-np.minimum(scores[stable_indices], flip_scores[flip_index[stable_indices]]))
            stable_indices = stable_indices[order[:CFG["MAX_ANCHORS_PER_IMAGE"]]]
        if len(stable_indices):
            anchors.append({
                "image_id": path.stem,
                "path": path,
                "candidate_indices": stable_indices.astype(np.int64),
                "boxes": boxes[stable_indices].copy(),
                "scores": scores[stable_indices].copy(),
                "legacy_poison_scores": poison_scores[stable_indices].copy(),
                "flip_iou": flip_iou[stable_indices].copy(),
            })
    total_anchors = sum(len(record["boxes"]) for record in anchors)
    print("anchor images:", len(anchors), "| real anchors:", total_anchors)
    assert total_anchors >= CFG["MIN_REAL_ANCHORS"], (
        f"only {total_anchors} real anchors; lower no gate automatically—inspect mining first"
    )
    return anchors, candidate_cache

def split_anchor_folds(test_paths, number_of_folds, seed):
    assignments = {}
    for path in test_paths:
        digest = hashlib.sha1(f"anchor:{seed}:{path.stem}".encode()).hexdigest()
        assignments[path.stem] = int(digest[:8], 16) % number_of_folds
    return assignments

test_files = sorted(Path(TEST_DIR).glob("*.png"), key=stem_key)
real_anchors, test_candidate_cache = mine_real_anchors(test_files, teacher, legacy_ranker)
ANCHOR_FOLD = split_anchor_folds(test_files, CFG["N_FOLDS"], CFG["SEED"])

anchor_counts = []
for fold_index in range(CFG["N_FOLDS"]):
    count = sum(
        len(record["boxes"])
        for record in real_anchors
        if ANCHOR_FOLD[record["image_id"]] == fold_index
    )
    anchor_counts.append(count)
    print(f"anchor fold {fold_index}: {count} held-out anchors")
assert min(anchor_counts) > 0, "every anchor validation fold must be non-empty"


mine real anchors:   0%|          | 0/2000 [00:00<?, ?it/s]

anchor images: 167 | real anchors: 179
anchor fold 0: 46 held-out anchors
anchor fold 1: 43 held-out anchors
anchor fold 2: 44 held-out anchors
anchor fold 3: 46 held-out anchors


In [8]:
def in_scrub_scope(name):
    return (
        name.startswith("head.")
        or name.startswith("backbone.fpn")
        or name.startswith("backbone.bottom_up.res5")
    )

def empty_detection_loss(model, images, iteration):
    records = [
        to_batched(image, np.zeros((0, 4), np.float32), train=True)
        for image in images
    ]
    with EventStorage(iteration):
        losses = model(records)
    return losses["loss_cls"] + losses.get("loss_box_reg", 0)

def streaming_topk_threshold(tensors, fraction, steps):
    values = list(tensors)
    total_values = int(sum(value.numel() for value in values))
    target_count = max(1, int(math.ceil(total_values * fraction)))
    _ = steps
    if not all(bool(torch.isfinite(value).all().item()) for value in values):
        return float("inf"), 0, 0
    global_top = None
    for value in values:
        local_count = min(target_count, value.numel())
        local_top = torch.topk(value.reshape(-1), local_count, sorted=False).values
        if global_top is None:
            global_top = local_top
        else:
            merged = torch.cat([global_top, local_top])
            keep_count = min(target_count, merged.numel())
            global_top = torch.topk(merged, keep_count, sorted=False).values
    assert global_top is not None and global_top.numel() == target_count
    threshold = float(global_top.min().item())
    del global_top
    if threshold <= 0.0:
        strict_selected = int(sum(
            torch.count_nonzero(value > 0.0).item() for value in values
        ))
        return 0.0, strict_selected, 0
    strict_selected = int(sum(
        torch.count_nonzero(value > threshold).item() for value in values
    ))
    tie_budget = target_count - strict_selected
    assert 0 <= tie_budget <= target_count
    return threshold, strict_selected, tie_budget

def build_budgeted_mask(tensors, threshold, tie_budget, chunk_size=1_000_000):
    mask = {
        name: (value > threshold).to(torch.uint8)
        for name, value in tensors.items()
    }
    remaining = int(tie_budget)
    if remaining <= 0 or not math.isfinite(threshold):
        return mask
    for name, value in tensors.items():
        flat_value = value.reshape(-1)
        flat_mask = mask[name].reshape(-1)
        for start in range(0, flat_value.numel(), chunk_size):
            if remaining <= 0:
                break
            end = min(start + chunk_size, flat_value.numel())
            tie_positions = torch.nonzero(
                flat_value[start:end] == threshold, as_tuple=False
            ).reshape(-1)
            take = min(remaining, tie_positions.numel())
            if take:
                flat_mask[start + tie_positions[:take]] = 1
                remaining -= take
        if remaining <= 0:
            break
    assert remaining == 0, "global top-k tie budget could not be allocated"
    return mask

def compute_contrastive_mask(model, allowed_sources, fold_index):
    scope_parameters = [
        (name, parameter)
        for name, parameter in model.named_parameters()
        if in_scrub_scope(name)
    ]
    assert scope_parameters, "empty scrub scope"
    parameter_list = [parameter for _, parameter in scope_parameters]
    repeat_masks = []
    repeat_stats = []
    for repeat_index in range(CFG["SAL_REPEATS"]):
        rng = np.random.default_rng(20000 + 1000 * fold_index + repeat_index)
        accumulation = {
            name: torch.zeros_like(parameter, device=DEVICE)
            for name, parameter in scope_parameters
        }
        model.train()
        for batch_index in range(CFG["SAL_BATCHES"]):
            pairs = [
                make_matched_pair(allowed_sources, rng)
                for _ in range(CFG["BS_FORGET"])
            ]
            poison_loss = empty_detection_loss(
                model, [pair["poison"] for pair in pairs], batch_index
            )
            poison_gradients = torch.autograd.grad(
                poison_loss, parameter_list, allow_unused=True
            )
            control_loss = empty_detection_loss(
                model, [pair["control"] for pair in pairs], batch_index
            )
            control_gradients = torch.autograd.grad(
                control_loss, parameter_list, allow_unused=True
            )
            for (name, parameter), poison_gradient, control_gradient in zip(
                scope_parameters, poison_gradients, control_gradients
            ):
                if poison_gradient is None:
                    poison_gradient = torch.zeros_like(parameter)
                if control_gradient is None:
                    control_gradient = torch.zeros_like(parameter)
                accumulation[name] += (poison_gradient - control_gradient).abs()
            model.zero_grad(set_to_none=True)
        threshold, strict_selected, tie_budget = streaming_topk_threshold(
            accumulation.values(), CFG["MASK_TOPK"], CFG["SAL_THRESHOLD_STEPS"]
        )
        current_mask = build_budgeted_mask(accumulation, threshold, tie_budget)
        selected_from_mask = int(sum(value.sum().item() for value in current_mask.values()))
        selected = strict_selected + tie_budget
        assert selected_from_mask == selected
        repeat_masks.append(current_mask)
        repeat_stats.append({
            "repeat": repeat_index,
            "threshold": float(threshold),
            "selected": selected,
        })
        del accumulation
        torch.cuda.empty_cache()

    pairwise_jaccard = []
    for first_index in range(len(repeat_masks)):
        for second_index in range(first_index + 1, len(repeat_masks)):
            intersection = 0
            union = 0
            for name, _ in scope_parameters:
                first = repeat_masks[first_index][name].bool()
                second = repeat_masks[second_index][name].bool()
                intersection += int((first & second).sum().item())
                union += int((first | second).sum().item())
            pairwise_jaccard.append(intersection / max(union, 1))
    stability = float(np.mean(pairwise_jaccard)) if pairwise_jaccard else 1.0
    stable_mask = {}
    for name, _ in scope_parameters:
        votes = torch.stack([mask[name] for mask in repeat_masks], dim=0).sum(dim=0)
        stable_mask[name] = (votes >= CFG["MASK_MIN_VOTES"]).float()
    selected_stable = int(sum(value.sum().item() for value in stable_mask.values()))
    total_scope = int(sum(value.numel() for value in stable_mask.values()))
    assert selected_stable > 0, "stable contrastive mask selected no weights"
    summary = {
        "fold": fold_index,
        "stability_jaccard": stability,
        "stable_selected": selected_stable,
        "scope_total": total_scope,
        "stable_fraction": selected_stable / max(total_scope, 1),
        "repeats": repeat_stats,
    }
    print("contrastive mask:", summary)
    return stable_mask, summary


In [9]:
def anchor_localized_kd(student, teacher_model, records):
    classification_loss = torch.zeros((), device=DEVICE)
    box_loss = torch.zeros((), device=DEVICE)
    selected_anchors = 0
    for record in records:
        target_boxes = record["anchor_boxes_resized"].to(DEVICE)
        if len(target_boxes) == 0:
            continue
        with torch.no_grad():
            teacher_logits, teacher_deltas, generated_anchors = head_tensors(teacher_model, [record])
        student_logits, student_deltas, _ = head_tensors(student, [record])
        overlaps = pairwise_iou(Boxes(generated_anchors), Boxes(target_boxes))
        maximum_iou = overlaps.max(dim=1).values
        teacher_probabilities = torch.sigmoid(teacher_logits[0, :, 0])
        selected = (
            (maximum_iou >= CFG["KD_ANCHOR_IOU"])
            & (teacher_probabilities >= CFG["KD_TEACHER_PROB"])
        )
        if not selected.any():
            fallback_count = max(1, int(math.ceil(maximum_iou.numel() * 0.005)))
            fallback_indices = torch.topk(maximum_iou, fallback_count).indices
            selected = torch.zeros_like(maximum_iou, dtype=torch.bool)
            selected[fallback_indices] = True
        selected_anchors += int(selected.sum().item())
        classification_loss = classification_loss + F.binary_cross_entropy_with_logits(
            student_logits[0, selected, 0],
            teacher_probabilities[selected],
            reduction="mean",
        )
        probability_weights = teacher_probabilities[selected].detach().unsqueeze(1)
        box_loss = box_loss + (
            probability_weights
            * (student_deltas[0, selected] - teacher_deltas[0, selected]).abs()
        ).sum() / (probability_weights.sum() * 4.0 + 1e-6)
    divisor = max(len(records), 1)
    return classification_loss / divisor, box_loss / divisor, selected_anchors

def anchor_records_for_training(anchor_pool, rng, batch_size):
    indices = rng.integers(0, len(anchor_pool), size=batch_size)
    records = []
    for index in indices:
        anchor = anchor_pool[int(index)]
        image = load_image(anchor["path"])
        records.append(to_batched(image, anchor["boxes"], train=False))
    return records

def target_confidence(model, image, target_box, match_iou=0.3):
    boxes, scores = infer_np(model, image)
    if len(boxes) == 0:
        return 0.0, None
    overlaps = iou_matrix(target_box, boxes)[0]
    valid = overlaps >= match_iou
    if not valid.any():
        return 0.0, None
    valid_indices = np.flatnonzero(valid)
    best = valid_indices[np.argmax(scores[valid_indices])]
    return float(scores[best]), boxes[best].copy()

def rank_auc_on_fold(scrub, held_pairs, held_anchors, candidate_cache):
    positive_new, positive_legacy = [], []
    detected_sources = set()
    for pair in held_pairs:
        image = pair["poison"]
        target = pair["target"]
        teacher_boxes, teacher_scores = infer_np(teacher, image)
        if len(teacher_boxes) == 0:
            continue
        overlap = iou_matrix(target, teacher_boxes)[0]
        teacher_index = int(overlap.argmax())
        if overlap[teacher_index] < 0.3 or teacher_scores[teacher_index] < 0.15:
            continue
        scrub_ratio = match_ratio(
            teacher_boxes,
            teacher_scores,
            *infer_np(scrub, image),
        )
        legacy_ratio = match_ratio(
            teacher_boxes,
            teacher_scores,
            *infer_np(legacy_ranker, image),
        )
        positive_new.append(1.0 - float(scrub_ratio[teacher_index]))
        positive_legacy.append(1.0 - float(legacy_ratio[teacher_index]))
        detected_sources.add(pair["source_id"])

    negative_new, negative_legacy = [], []
    for anchor in held_anchors:
        cached = candidate_cache[anchor["image_id"]]
        scrub_ratio = match_ratio(
            cached["boxes"], cached["scores"], *infer_np(scrub, load_image(anchor["path"]))
        )
        for candidate_index, legacy_score in zip(
            anchor["candidate_indices"], anchor["legacy_poison_scores"]
        ):
            negative_new.append(1.0 - float(scrub_ratio[int(candidate_index)]))
            negative_legacy.append(float(legacy_score))

    labels = np.concatenate([
        np.ones(len(positive_new), np.int64),
        np.zeros(len(negative_new), np.int64),
    ])
    new_values = np.asarray(positive_new + negative_new, np.float32)
    legacy_values = np.asarray(positive_legacy + negative_legacy, np.float32)
    blended_values = 0.5 * new_values + 0.5 * legacy_values
    if len(positive_new) < 4 or len(negative_new) < 4:
        return {
            "new_auc": 0.0, "legacy_auc": 0.0, "blend_auc": 0.0,
            "source_coverage": 0.0, "rank_positive_n": len(positive_new),
            "rank_negative_n": len(negative_new),
        }
    return {
        "new_auc": float(roc_auc_score(labels, new_values)),
        "legacy_auc": float(roc_auc_score(labels, legacy_values)),
        "blend_auc": float(roc_auc_score(labels, blended_values)),
        "source_coverage": len(detected_sources) / max(len(set(pair["source_id"] for pair in held_pairs)), 1),
        "rank_positive_n": len(positive_new),
        "rank_negative_n": len(negative_new),
    }


In [10]:
def evaluate_fold(scrub, held_pairs, held_anchors, candidate_cache, include_rank=True):
    poison_teacher = []
    poison_scrub = []
    detected_sources = set()
    for pair in held_pairs:
        teacher_confidence, _ = target_confidence(teacher, pair["poison"], pair["target"])
        scrub_confidence, _ = target_confidence(scrub, pair["poison"], pair["target"])
        poison_teacher.append(teacher_confidence)
        poison_scrub.append(scrub_confidence)
        if teacher_confidence >= 0.15:
            detected_sources.add(pair["source_id"])

    confidence_ratios = []
    recalls = []
    matched_ious = []
    for anchor in held_anchors:
        image = load_image(anchor["path"])
        scrub_boxes, scrub_scores = infer_np(scrub, image)
        overlaps = iou_matrix(anchor["boxes"], scrub_boxes)
        for anchor_index, teacher_score in enumerate(anchor["scores"]):
            if overlaps.shape[1] == 0:
                confidence_ratios.append(0.0)
                recalls.append(0.0)
                matched_ious.append(0.0)
                continue
            best_index = int(overlaps[anchor_index].argmax())
            best_iou = float(overlaps[anchor_index, best_index])
            matched_ious.append(best_iou)
            if best_iou >= 0.5:
                recalls.append(1.0)
                confidence_ratios.append(min(
                    1.5,
                    float(scrub_scores[best_index]) / max(float(teacher_score), 1e-6),
                ))
            else:
                recalls.append(0.0)
                confidence_ratios.append(0.0)

    rank = {
        "new_auc": 0.0, "legacy_auc": 0.0, "blend_auc": 0.0,
        "source_coverage": len(detected_sources) / max(len(set(pair["source_id"] for pair in held_pairs)), 1),
        "rank_positive_n": 0, "rank_negative_n": 0,
    }
    if include_rank:
        rank = rank_auc_on_fold(scrub, held_pairs, held_anchors, candidate_cache)
    teacher_mean = float(np.mean(poison_teacher)) if poison_teacher else 0.0
    scrub_mean = float(np.mean(poison_scrub)) if poison_scrub else 1.0
    metrics = {
        "poison_teacher_conf": teacher_mean,
        "poison_scrub_conf": scrub_mean,
        "poison_reduction": teacher_mean - scrub_mean,
        "real_conf_ratio": float(np.mean(confidence_ratios)) if confidence_ratios else 0.0,
        "real_recall": float(np.mean(recalls)) if recalls else 0.0,
        "real_iou": float(np.mean(matched_ious)) if matched_ious else 0.0,
        "held_source_count": len(set(pair["source_id"] for pair in held_pairs)),
        "detected_source_count": len(detected_sources),
        "held_anchor_count": int(sum(len(anchor["boxes"]) for anchor in held_anchors)),
    }
    metrics.update(rank)
    return metrics

def checkpoint_score(metrics):
    poison_term = metrics["poison_teacher_conf"] - metrics["poison_scrub_conf"]
    return (
        poison_term
        + 0.25 * metrics["real_conf_ratio"]
        + 0.25 * metrics["real_recall"]
        + 0.10 * metrics["real_iou"]
    )

def checkpoint_is_admissible(metrics):
    return (
        metrics["real_conf_ratio"] >= CFG["REAL_CONF_MIN"]
        and metrics["real_recall"] >= CFG["REAL_RECALL_MIN"]
        and metrics["real_iou"] >= CFG["REAL_IOU_MIN"]
    )

def train_scrub_fold(
    fold_index, training_sources, training_anchors, checkpoint_pairs, checkpoint_anchors
):
    scrub = load_model()
    for name, parameter in scrub.named_parameters():
        parameter.requires_grad_(in_scrub_scope(name))
    trainable = [parameter for parameter in scrub.parameters() if parameter.requires_grad]
    stable_mask, mask_summary = compute_contrastive_mask(scrub, training_sources, fold_index)
    original_weights = {
        name: parameter.detach().clone()
        for name, parameter in scrub.named_parameters()
        if parameter.requires_grad
    }
    optimizer = torch.optim.AdamW(trainable, lr=CFG["SCRUB_LR"], weight_decay=0.0)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, CFG["SCRUB_ITERS"])
    rng = np.random.default_rng(30000 + fold_index)
    best_any = {"score": -1e9, "iteration": -1, "state": None, "metrics": None}
    best_safe = {"score": -1e9, "iteration": -1, "state": None, "metrics": None}
    start_time = time.time()
    for iteration in range(1, CFG["SCRUB_ITERS"] + 1):
        scrub.train()
        forget_pairs = [
            make_matched_pair(training_sources, rng)
            for _ in range(CFG["BS_FORGET"])
        ]
        poison_loss = empty_detection_loss(
            scrub, [pair["poison"] for pair in forget_pairs], iteration
        )
        forget_loss = poison_loss
        retain_records = anchor_records_for_training(
            training_anchors,
            rng,
            CFG["BS_RETAIN"],
        )
        kd_cls, kd_box, selected_anchor_count = anchor_localized_kd(
            scrub, teacher, retain_records
        )
        l2_sp = torch.zeros((), device=DEVICE)
        for name, parameter in scrub.named_parameters():
            if parameter.requires_grad:
                l2_sp = l2_sp + ((parameter - original_weights[name]) ** 2).sum()
        total_loss = (
            CFG["W_FORGET"] * forget_loss
            + CFG["W_KD"] * kd_cls
            + CFG["W_KD_BOX"] * kd_box
            + CFG["W_SP"] * l2_sp
        )
        optimizer.zero_grad(set_to_none=True)
        total_loss.backward()
        for name, parameter in scrub.named_parameters():
            if parameter.requires_grad and parameter.grad is not None:
                parameter.grad.mul_(stable_mask[name])
        torch.nn.utils.clip_grad_norm_(trainable, CFG["GRAD_CLIP"])
        optimizer.step()
        scheduler.step()

        if iteration % CFG["EVAL_EVERY"] == 0 or iteration == CFG["SCRUB_ITERS"]:
            scrub.eval()
            metrics = evaluate_fold(
                scrub,
                checkpoint_pairs,
                checkpoint_anchors,
                test_candidate_cache,
                include_rank=False,
            )
            score = checkpoint_score(metrics)
            print(
                f"fold={fold_index} iter={iteration:4d} "
                f"forget={float(forget_loss.detach()):.4f} "
                f"kd={float(kd_cls.detach()):.4f}/{float(kd_box.detach()):.4f} "
                f"anchors={selected_anchor_count} supp={metrics['poison_scrub_conf']:.3f} "
                f"conf={metrics['real_conf_ratio']:.3f} recall={metrics['real_recall']:.3f} "
                f"iou={metrics['real_iou']:.3f} elapsed={time.time()-start_time:.0f}s"
            )
            checkpoint = {
                "score": score,
                "iteration": iteration,
                "metrics": metrics,
                "state": {
                    name: value.detach().cpu().clone()
                    for name, value in scrub.state_dict().items()
                },
            }
            accepted = False
            if score > best_any["score"]:
                best_any = checkpoint
                accepted = True
            if checkpoint_is_admissible(metrics) and score > best_safe["score"]:
                best_safe = checkpoint
                accepted = True
            if not accepted:
                checkpoint = None
    chosen = best_safe if best_safe["state"] is not None else best_any
    assert chosen["state"] is not None
    scrub.load_state_dict(chosen["state"])
    chosen["state"] = None
    scrub.eval()
    checkpoint_path = WORK / f"nb49_fold{fold_index}_scrub.pth"
    torch.save({"model": scrub.state_dict()}, checkpoint_path)
    print(
        "restored fold checkpoint:", fold_index, chosen["iteration"], chosen["metrics"],
        "safe=", best_safe["iteration"] >= 0,
    )
    return scrub, mask_summary, chosen, checkpoint_path


In [11]:
fold_checkpoints = []
fold_rows = []
fold_artifacts = []

for fold_index in range(CFG["N_FOLDS"]):
    gate_source_fold = fold_index
    checkpoint_source_fold = (fold_index + 1) % CFG["N_FOLDS"]
    gate_anchor_fold = fold_index
    checkpoint_anchor_fold = (fold_index + 1) % CFG["N_FOLDS"]
    held_sources = SOURCE_FOLDS[gate_source_fold]
    checkpoint_sources = SOURCE_FOLDS[checkpoint_source_fold]
    training_sources = sorted(
        set(source_ids) - set(held_sources) - set(checkpoint_sources)
    )
    checkpoint_pairs = make_validation_pairs(
        checkpoint_sources, fold_index, count=16, seed_offset=0
    )
    gate_pairs = make_validation_pairs(
        held_sources, fold_index, count=32, seed_offset=1
    )
    training_anchors = [
        anchor for anchor in real_anchors
        if ANCHOR_FOLD[anchor["image_id"]]
        not in {gate_anchor_fold, checkpoint_anchor_fold}
    ]
    checkpoint_anchors = [
        anchor for anchor in real_anchors
        if ANCHOR_FOLD[anchor["image_id"]] == checkpoint_anchor_fold
    ]
    gate_anchors = [
        anchor for anchor in real_anchors
        if ANCHOR_FOLD[anchor["image_id"]] == gate_anchor_fold
    ]
    assert training_sources and training_anchors
    assert checkpoint_anchors and gate_anchors, "need disjoint checkpoint and gate anchor images"
    assert not set(anchor["image_id"] for anchor in checkpoint_anchors) & set(
        anchor["image_id"] for anchor in gate_anchors
    )

    scrub, mask_summary, best, checkpoint_path = train_scrub_fold(
        fold_index,
        training_sources,
        training_anchors,
        checkpoint_pairs,
        checkpoint_anchors,
    )
    gate_metrics = evaluate_fold(scrub, gate_pairs, gate_anchors, test_candidate_cache)
    fold_row = {
        "fold": fold_index,
        "training_sources": training_sources,
        "checkpoint_sources": checkpoint_sources,
        "held_sources": held_sources,
        "checkpoint_pair_seed": 10000 + 100 * fold_index,
        "gate_pair_seed": 10001 + 100 * fold_index,
        "mask_stability": mask_summary["stability_jaccard"],
        "mask_fraction": mask_summary["stable_fraction"],
        "checkpoint_iteration": best["iteration"],
        "checkpoint_path": str(checkpoint_path),
        **gate_metrics,
    }
    print("GATE METRICS:", fold_row)
    fold_rows.append(fold_row)
    fold_checkpoints.append(checkpoint_path)
    scrub.to("cpu")
    del scrub
    torch.cuda.empty_cache()
    best_summary = {
        "score": best["score"],
        "iteration": best["iteration"],
        "metrics": best["metrics"],
    }
    fold_artifacts.append({
        "fold": fold_index,
        "mask": mask_summary,
        "checkpoint": best_summary,
        "gate": gate_metrics,
    })

fold_frame = pd.DataFrame(fold_rows)
fold_frame.to_csv(WORK / "nb49_fold_metrics.csv", index=False)
with open(WORK / "nb49_fold_artifacts.json", "w") as handle:
    json.dump(fold_artifacts, handle, indent=2, default=str)
print(fold_frame.to_string(index=False))


contrastive mask: {'fold': 0, 'stability_jaccard': 0.6210575684175784, 'stable_selected': 1067099, 'scope_total': 22432035, 'stable_fraction': 0.04757031629096513, 'repeats': [{'repeat': 0, 'threshold': 0.01184600219130516, 'selected': 1121602}, {'repeat': 1, 'threshold': 0.05941447988152504, 'selected': 1121602}, {'repeat': 2, 'threshold': 0.2112194299697876, 'selected': 1121602}]}
fold=0 iter=  50 forget=0.0233 kd=0.3537/0.0076 anchors=40 supp=0.254 conf=0.928 recall=1.000 iou=0.975 elapsed=46s
fold=0 iter= 100 forget=0.0037 kd=0.3454/0.0038 anchors=40 supp=0.205 conf=0.936 recall=1.000 iou=0.978 elapsed=93s
fold=0 iter= 150 forget=0.0022 kd=0.3101/0.0029 anchors=35 supp=0.197 conf=0.976 recall=1.000 iou=0.985 elapsed=139s
fold=0 iter= 200 forget=0.0087 kd=0.4104/0.0027 anchors=32 supp=0.186 conf=0.967 recall=1.000 iou=0.982 elapsed=185s
fold=0 iter= 250 forget=0.0123 kd=0.3343/0.0025 anchors=41 supp=0.189 conf=0.976 recall=1.000 iou=0.986 elapsed=231s
fold=0 iter= 300 forget=0.0022 

In [12]:
def source_sets_are_disjoint(row):
    training = set(row["training_sources"])
    checkpoint = set(row["checkpoint_sources"])
    held = set(row["held_sources"])
    return (
        not (training & checkpoint)
        and not (training & held)
        and not (checkpoint & held)
        and training | checkpoint | held == set(source_ids)
    )

SOURCE_DISJOINT_PASS = bool(
    all(source_sets_are_disjoint(row) for row in fold_rows)
    and all(row["source_coverage"] >= 1.0 for row in fold_rows)
    and all(row["detected_source_count"] == row["held_source_count"] for row in fold_rows)
)
MASK_STABILITY_PASS = bool(
    all(
        row["mask_stability"] >= CFG["MASK_STABILITY_MIN"]
        and row["mask_fraction"] <= CFG["MASK_FRACTION_MAX"]
        for row in fold_rows
    )
)
POISON_SUPPRESSION_PASS = bool(all(
    row["poison_scrub_conf"] <= CFG["SUPP_MAX"]
    and row["poison_reduction"] >= CFG["SUPP_REDUCTION_MIN"]
    for row in fold_rows
))
REAL_CONFIDENCE_PASS = bool(all(
    row["real_conf_ratio"] >= CFG["REAL_CONF_MIN"]
    and row["real_recall"] >= CFG["REAL_RECALL_MIN"]
    for row in fold_rows
))
BOX_FIDELITY_PASS = bool(all(
    row["real_iou"] >= CFG["REAL_IOU_MIN"] for row in fold_rows
))
mean_legacy_auc = float(np.mean([row["legacy_auc"] for row in fold_rows]))
mean_blend_auc = float(np.mean([row["blend_auc"] for row in fold_rows]))
RANK_AUC_PASS = bool(
    all(
        row["blend_auc"] >= CFG["RANK_AUC_MIN"]
        and row["blend_auc"] + CFG["RANK_FOLD_TOLERANCE"] >= row["legacy_auc"]
        for row in fold_rows
    )
    and mean_blend_auc >= mean_legacy_auc + CFG["RANK_AUC_GAIN_MIN"]
)

ALL_GATES_PASS = bool(
    CFG["RULES_CONFIRMED"]
    and SOURCE_DISJOINT_PASS
    and MASK_STABILITY_PASS
    and POISON_SUPPRESSION_PASS
    and REAL_CONFIDENCE_PASS
    and BOX_FIDELITY_PASS
    and RANK_AUC_PASS
)

gate_report = {
    "RULES_CONFIRMED": bool(CFG["RULES_CONFIRMED"]),
    "SOURCE_DISJOINT_PASS": SOURCE_DISJOINT_PASS,
    "MASK_STABILITY_PASS": MASK_STABILITY_PASS,
    "POISON_SUPPRESSION_PASS": POISON_SUPPRESSION_PASS,
    "REAL_CONFIDENCE_PASS": REAL_CONFIDENCE_PASS,
    "BOX_FIDELITY_PASS": BOX_FIDELITY_PASS,
    "RANK_AUC_PASS": RANK_AUC_PASS,
    "ALL_GATES_PASS": ALL_GATES_PASS,
    "rank_summary": {
        "mean_legacy_auc": mean_legacy_auc,
        "mean_blend_auc": mean_blend_auc,
        "mean_gain": mean_blend_auc - mean_legacy_auc,
    },
    "thresholds": {
        key: CFG[key]
        for key in (
            "MASK_STABILITY_MIN", "SUPP_MAX", "SUPP_REDUCTION_MIN",
            "REAL_CONF_MIN", "REAL_RECALL_MIN", "REAL_IOU_MIN",
            "RANK_AUC_MIN", "RANK_AUC_GAIN_MIN", "RANK_FOLD_TOLERANCE",
        )
    },
    "folds": fold_rows,
}
with open(WORK / "nb49_gate_report.json", "w") as handle:
    json.dump(gate_report, handle, indent=2, default=str)

print("\n=== NB49 HARD GATES ===")
print(
    f"rank mean legacy/blend/gain: {mean_legacy_auc:.6f} / "
    f"{mean_blend_auc:.6f} / {mean_blend_auc - mean_legacy_auc:+.6f}"
)
for gate_name in (
    "SOURCE_DISJOINT_PASS", "MASK_STABILITY_PASS", "POISON_SUPPRESSION_PASS",
    "REAL_CONFIDENCE_PASS", "BOX_FIDELITY_PASS", "RANK_AUC_PASS",
    "ALL_GATES_PASS",
):
    print(f"{gate_name:28s}: {gate_report[gate_name]}")



=== NB49 HARD GATES ===
rank mean legacy/blend/gain: 0.988636 / 0.999012 / +0.010375
SOURCE_DISJOINT_PASS        : True
MASK_STABILITY_PASS         : True
POISON_SUPPRESSION_PASS     : True
REAL_CONFIDENCE_PASS        : True
BOX_FIDELITY_PASS           : True
RANK_AUC_PASS               : True
ALL_GATES_PASS              : True


In [13]:
def aggregate_crossfit_score(test_paths, candidate_cache, checkpoint_paths):
    score_by_image = {}
    component_rows = []
    for fold_index, checkpoint_path in enumerate(checkpoint_paths):
        fold_model = load_model(checkpoint_path)
        fold_paths = [
            path for path in test_paths if ANCHOR_FOLD[path.stem] == fold_index
        ]
        for path in tqdm(fold_paths, desc=f"cross-fit rank fold {fold_index}"):
            cache = candidate_cache[path.stem]
            scrub_boxes, scrub_scores = infer_np(fold_model, load_image(path))
            scrub_ratio = match_ratio(
                cache["boxes"], cache["scores"], scrub_boxes, scrub_scores
            )
            legacy_poison = 1.0 - cache["legacy_ratio"]
            scrub_poison = 1.0 - scrub_ratio
            blended_poison = (
                0.5 * legacy_poison + 0.5 * scrub_poison
            ).astype(np.float32)
            score_by_image[path.stem] = blended_poison
            component_rows.append({
                "image_id": path.stem,
                "fold": fold_index,
                "candidate_count": len(blended_poison),
                "legacy_mean": float(legacy_poison.mean()) if len(legacy_poison) else 0.0,
                "scrub_mean": float(scrub_poison.mean()) if len(scrub_poison) else 0.0,
                "blend_mean": float(blended_poison.mean()) if len(blended_poison) else 0.0,
            })
        fold_model.to("cpu")
        del fold_model
        torch.cuda.empty_cache()
    assert set(score_by_image) == set(candidate_cache), "cross-fit score coverage is incomplete"
    return score_by_image, pd.DataFrame(component_rows)

def normalize_id(value):
    text = str(value)
    try:
        return str(int(text))
    except ValueError:
        return text

def format_prediction(boxes, scores):
    values = []
    for (x1, y1, x2, y2), score in zip(boxes, scores):
        x1 = float(np.clip(x1, 0, 1024))
        y1 = float(np.clip(y1, 0, 1024))
        x2 = float(np.clip(x2, 0, 1024))
        y2 = float(np.clip(y2, 0, 1024))
        width, height = x2 - x1, y2 - y1
        if width > 0 and height > 0 and score > 0:
            values.extend([
                f"{float(score):.6f}", f"{x1:.2f}", f"{y1:.2f}",
                f"{width:.2f}", f"{height:.2f}",
            ])
    return " ".join(values) if values else " "

def exact_keep_count(score_by_image, threshold):
    total = 0
    for stem, cache in test_candidate_cache.items():
        eligible = cache["scores"] >= CFG["BASE_THR"]
        scores = cache["scores"][eligible]
        poison = score_by_image[stem][eligible]
        keep = (scores >= CFG["S_MIN"]) & (poison < threshold)
        total += min(int(keep.sum()), CFG["MAX_KEEP_PER_IMAGE"])
    return total

def calibrate_rank_threshold(score_by_image):
    available = []
    for stem, cache in test_candidate_cache.items():
        eligible = (cache["scores"] >= CFG["BASE_THR"]) & (cache["scores"] >= CFG["S_MIN"])
        available.append(score_by_image[stem][eligible])
    available = np.concatenate(available) if available else np.array([0.5], np.float32)
    grid = np.unique(np.quantile(available, np.linspace(0.01, 0.99, 160)))
    candidates = [
        (abs(exact_keep_count(score_by_image, threshold) - CFG["TARGET_K"]),
         float(threshold),
         exact_keep_count(score_by_image, threshold))
        for threshold in grid
    ]
    _, threshold, keep_count = min(candidates, key=lambda item: (item[0], item[1]))
    return threshold, keep_count

def compose_crossfit_submission(score_by_image, threshold, output_path):
    predictions = {}
    total_keep = 0
    boosted_mass = 0.0
    for stem, cache in test_candidate_cache.items():
        eligible_indices = np.flatnonzero(cache["scores"] >= CFG["BASE_THR"])
        boxes = cache["boxes"][eligible_indices]
        teacher_scores = cache["scores"][eligible_indices]
        poison_scores = score_by_image[stem][eligible_indices]
        output_scores = np.full(len(eligible_indices), CFG["EPS_Q"], np.float32)
        keep = (teacher_scores >= CFG["S_MIN"]) & (poison_scores < threshold)
        if keep.sum() > CFG["MAX_KEEP_PER_IMAGE"]:
            eligible_keep = np.flatnonzero(keep)
            order = np.argsort(poison_scores[eligible_keep])
            allowed = eligible_keep[order[:CFG["MAX_KEEP_PER_IMAGE"]]]
            reduced = np.zeros_like(keep, dtype=bool)
            reduced[allowed] = True
            keep &= reduced
        output_scores[keep] = np.minimum(1.0, teacher_scores[keep] + CFG["BOOST"])
        total_keep += int(keep.sum())
        boosted_mass += float(output_scores[keep].sum())
        predictions[normalize_id(stem)] = (boxes, output_scores)
    sample = pd.read_csv(SAMPLE_SUB, dtype={"image_id": str})
    sample["prediction_string"] = sample["image_id"].map(
        lambda image_id: format_prediction(*predictions.get(
            normalize_id(image_id),
            (np.zeros((0, 4), np.float32), np.zeros(0, np.float32)),
        ))
    )
    sample.to_csv(output_path, index=False)
    return {"path": str(output_path), "keepK": total_keep, "boosted_mass": boosted_mass}

export_path = WORK / "nb49_crossfit_rank_Kcal.csv"
export_path.unlink(missing_ok=True)
if ALL_GATES_PASS:
    crossfit_scores, crossfit_components = aggregate_crossfit_score(
        test_files, test_candidate_cache, fold_checkpoints
    )
    crossfit_components.to_csv(WORK / "nb49_crossfit_components.csv", index=False)
    calibrated_threshold, calibrated_keep = calibrate_rank_threshold(crossfit_scores)
    export_manifest = compose_crossfit_submission(
        crossfit_scores, calibrated_threshold, export_path
    )
    export_manifest.update({
        "threshold": calibrated_threshold,
        "calibrated_keep": calibrated_keep,
        "target_keep": CFG["TARGET_K"],
        "policy": "teacher candidates + 0.5 legacy 1-r + 0.5 cross-fit scrub 1-r",
    })
    with open(WORK / "nb49_export_manifest.json", "w") as handle:
        json.dump(export_manifest, handle, indent=2)
    print("NB49 gated export:", export_manifest)
else:
    print("NB49 FAIL CLOSED: no nb49_crossfit_rank_Kcal.csv was created.")
    print("Keep the proven 205.9359 submission; do not submit a failed-gate repair.")

print("NEVER auto-submit: inspect nb49_gate_report.json and every fold before using the CSV.")


cross-fit rank fold 0:   0%|          | 0/517 [00:00<?, ?it/s]

cross-fit rank fold 1:   0%|          | 0/483 [00:00<?, ?it/s]

cross-fit rank fold 2:   0%|          | 0/476 [00:00<?, ?it/s]

cross-fit rank fold 3:   0%|          | 0/524 [00:00<?, ?it/s]

NB49 gated export: {'path': '/kaggle/working/nb49_crossfit_rank_Kcal.csv', 'keepK': 466, 'boosted_mass': 393.0382249355316, 'threshold': 0.37746330567875747, 'calibrated_keep': 466, 'target_keep': 467, 'policy': 'teacher candidates + 0.5 legacy 1-r + 0.5 cross-fit scrub 1-r'}
NEVER auto-submit: inspect nb49_gate_report.json and every fold before using the CSV.


## Decision policy

- `ALL_GATES_PASS=False`: stop. Do not tune thresholds until a failed scientific gate turns green.
- `ALL_GATES_PASS=True`: the CSV is only a leaderboard experiment, not evidence of LB 139.
- Submit the ranker before considering any repaired source model. This notebook intentionally has no
  source-model composer.
- Keep the proven LB 205.9359 file available as the fallback.
